In [1]:
import os
from typing import Literal
from pydantic import BaseModel
from tavily import TavilyClient
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from dotenv import load_dotenv
load_dotenv()

C:\Users\Deepam\AppData\Local\Temp\ipykernel_10572\118620161.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# 1. Load the PDF
pdf_loader = PyPDFLoader("../season11/evs_oil_price_shock.pdf")
raw_docs = pdf_loader.load()

# 2. Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = text_splitter.split_documents(raw_docs)
print(f"Number of chunks created: {len(chunks)}")

# 3. Wrap embeddings correctly
embeddings = SentenceTransformerEmbeddings(model_name="all-mpnet-base-v2")  # 768-dim

# 4. Create Chroma vectorstore
vectorstore = Chroma(
    persist_directory="./chroma_db",
    collection_name="rag_base_v2_mpnet",  # new collection for 768-dim
    embedding_function=embeddings
)

# 5. Add documents
vectorstore.add_documents(chunks)


Number of chunks created: 51


C:\Users\Deepam\AppData\Local\Temp\ipykernel_10572\4202819711.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-mpnet-base-v2")  # 768-dim


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

['cc15f042-09af-44cf-8aea-e7619dfb792f',
 'baa5e214-4f20-4315-aaeb-24dd1736527b',
 '228a1977-5d12-4bd4-b542-9999c573af72',
 '937dc735-ba77-4c92-bf02-f8bbd041086e',
 '9a0688ed-ef71-4c57-9e29-56e24fd8571b',
 'd5a224aa-c410-41ba-85a7-32da6777919d',
 '9c6334cf-968a-42a8-a182-0aeeac89c86d',
 'bee3975a-f7c0-41d8-a208-989490a77107',
 'b0146e58-9e9b-48ba-bc3c-bf90aeb12df9',
 'b065fc20-b951-469d-a2de-c069c35b80e1',
 '6d1a8ad6-2a07-4504-92fa-f34d16c2f237',
 '736b71f1-9879-4d7a-af2e-957bb9e17b3f',
 'cbd537d1-f2f3-45b8-8afb-5cb4e00e4eff',
 '553c5c1f-9ecf-4d33-b745-2ddfaad585b4',
 '7469376d-56bb-4ac4-8f76-cf0cc8c75a1d',
 '4c736d59-2510-4785-9eb0-5a5088dadb0e',
 '55f5993b-60fb-4155-83fb-84ed1e225095',
 'a817ee20-2fc0-47d0-a9b3-76274ae8a654',
 '6545dffc-69f0-459c-bc0a-5344b8afa1a6',
 '84ca7378-89d9-427c-9147-b891fc4a0e60',
 '416defc2-6687-4553-8cd0-3c4e3dd1523c',
 'f8ece481-70fc-42a5-bbd1-61d87c66a2bf',
 '31273855-d9a0-4ffb-a4f5-3c2c349f4755',
 '30929aaf-1d88-42bf-8ba7-9b0d366adeaa',
 '57078a78-660e-

In [3]:
class AgenticRAGState(MessagesState):
    query : str
    retrieved_docs : list[Document]
    relevant_doc : list[Document]
    context : str
    generation : str
    needs_retrieval : bool
    retrieval_count : int
    max_retrieval_step :int
    is_relevant :bool
    rewritten_query :str
    rewrite_count : int
    need_decomposistion :bool

class RouteDecision(BaseModel):
    needs_retrieval: bool

class RelevanceEvaluation(BaseModel):
    is_relevant: bool

class DecompositionDecision(BaseModel):
    needs_decomposition : bool

class QueryDecomposition(BaseModel):
    needs_decomposition: bool
    sub_queries: list[str]

In [5]:

# define the tools
@tool(response_format="content_and_artifact")
def vector_store_search(query: str, k: int = 3):
    """Search the vector store for relevant document passages.
    Adjust k (default 3) to retrieve more or fewer passages."""
    
    docs = vectorstore.similarity_search(query, k=k)
    context = "\n\n## Vector Store Results\n\n" + "\n\n".join(d.page_content for d in docs)
    
    return context, docs


@tool(response_format="content_and_artifact")
def web_search(query: str, max_results: int = 3):
    """Search the web for current or real-time information.
    Adjust max_results (default 3) to control how many results are returned."""
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    
    response = client.search(query, max_results=max_results)
    
    docs = [
        Document(
            page_content=r["content"],
            metadata={"source": r["url"], "title": r.get("title", "")},
        )
        for r in response["results"]
    ]
    
    content = "\n\n## Web Search Results\n\n" + "\n\n".join(d.page_content for d in docs)
    return content, docs

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from tool_use import vector_store_search # Since this tool is defined in tool_use.py

# 1. Initialize the LLM directly here
agent_llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro") 

# 2. Bind with tools
# Note: I'm leaving out `web_search` for now unless you have defined it in another cell
tools = [vector_store_search] 
agent_llm_with_tools = agent_llm.bind_tools(tools)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [ ]:
def check_decomposition(state: AgenticRAGState):
    prompt_template = ChatPromptTemplate([
          (
          "system",
            "You are a query analysis assistant. Determine if the user's question asks for "
            "multiple distinct pieces of information that each require separate retrieval. "
            "If yes, needs_decomposition should be True. "
            "IIf the question is simple or self-contained, needs_decomposition should be False.""
    ),
    ("human", "{query}"),
    ])
chain = prompt_template | llm.with_Str


In [9]:
query_1 = (
    "According to the report, by how many million barrels per day could a 300-million EV fleet "
    "displace oil demand by 2030, and what does this mean relative to total global demand? and what is current temp in mumbai"
)

In [ ]:
check_decomposition()